In [ ]:
import json 
# ── Helper: read JSONL log file ──────────────────────────────────────────────
def read_log(path):
    """Read a JSONL training log file. Returns a list of dicts (one per epoch)."""
    entries = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                entries.append(json.loads(line))
    entries.sort(key=lambda e: e['epoch'])
    return entries


def extract_train_losses(entries, keys=None):
    """Extract per-epoch training losses.
    Returns dict of {key: [values_per_epoch]} and list of epochs.
    """
    if keys is None:
        keys = ['train_loss', 'train_loss_ce', 'train_loss_bbox', 'train_loss_giou']
    epochs = [e['epoch'] for e in entries]
    return epochs, {k: [e[k] for e in entries] for k in keys}

def extract_test_losses(entries, heads, keys=None, head_key_prefix='test_heads_'):
    """Extract per-epoch test losses for a specific head count.
    Returns dict of {key: [values_per_epoch]} and list of epochs.
    """
    if keys is None:
        keys = ['loss', 'loss_ce', 'loss_bbox', 'loss_giou']
    head_key = f'{head_key_prefix}{heads}'
    epochs = [e['epoch'] for e in entries if head_key in e]
    return epochs, {k: [e[head_key][k] for e in entries if head_key in e] for k in keys}


def extract_test_ap(entries, heads, ap_index=0, head_key_prefix='test_heads_'):
    """Extract per-epoch AP from coco_eval_bbox for a specific head count.
    ap_index: 0=AP@[.50:.95], 1=AP@.50, etc.
    """
    head_key = f'{head_key_prefix}{heads}'
    epochs = [e['epoch'] for e in entries if head_key in e]
    values = [e[head_key]['coco_eval_bbox'][ap_index] for e in entries if head_key in e]
    return epochs, values

In [ ]:

import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path


# ── Plotting helpers ─────────────────────────────────────────────────────────
def plot_loss_comparison(logs, labels, colors, title='Training Loss',
                         loss_key='train_loss', figsize=(12, 5)):
    """Plot a single loss key across multiple log files."""
    fig, ax = plt.subplots(figsize=figsize)
    for entries, label, color in zip(logs, labels, colors):
        epochs = [e['epoch'] for e in entries]
        values = [e[loss_key] for e in entries]
        ax.plot(epochs, values, label=label, color=color, linewidth=1.5)
    ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax.set_ylabel(loss_key, fontsize=12, fontweight='bold')
    #ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    return fig, ax


# ── Load logs ────────────────────────────────────────────────────────────────
RESULTS = Path(r'C:\workspace\ml\workspace\master\original\results')
scaled_log  = read_log(RESULTS / 'layer_scaling' / '300epoch' / 'log.txt')
scaled_log_600  = read_log(RESULTS / 'layer_scaling' / '600epoch' / 'log.txt')
sliced_log  = read_log(RESULTS / 'sliced' / '300epoch' / 'log.txt')
sliced_log_600  = read_log(RESULTS / 'sliced' / '600epoch' / 'log.txt')

logs   = [sliced_log + sliced_log_600, scaled_log + scaled_log_600]
labels = ['Width Scalable-DETR', 'Depth Scalable-DETR']
colors = ['#C0392B', '#1E8449']

print(f"Sliced: {len(sliced_log)} epochs ({sliced_log[0]['epoch']}–{sliced_log[-1]['epoch']})")

# ── Training loss ────────────────────────────────────────────────────────────
plot_loss_comparison(logs, labels, colors, title='Training Loss Comparison', loss_key='train_loss')
plt.show()

In [ ]:
HEAD_MAPPING = {
    1: 'Ti',
    4: 'L',
}

LAYER_MAPPING = {
    1: 'Ti',
    5: 'L',
    6: 'XL',
}

fig, ax = plt.subplots(figsize=(12, 5))
for entries, label, color in zip(logs, labels, colors):
    if label == 'Depth Scalable-DETR':
        for h, ls in zip([1, 6], ['-', '--', ':']):
            epochs, ap = extract_test_ap(entries, h, head_key_prefix='test_layers_')
            ax.plot(epochs, ap, color=color, linestyle=ls, linewidth=1.5,
                    label=f'{label} — {LAYER_MAPPING.get(h, h)}')
    else: 
        for h, ls in zip([1, 4], ['-', '--']):
            epochs, ap = extract_test_ap(entries, h)
            ax.plot(epochs, ap, color=color, linestyle=ls, linewidth=1.5,
                    label=f'{label} — {HEAD_MAPPING.get(h, h)}')

ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax.set_ylabel('AP@[.50:.95]', fontsize=12, fontweight='bold')
#ax.set_title('AP@[.50:.95] over Training', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
norm_log  = read_log(RESULTS / 'norm' / 'log.txt')

logs   = [sliced_log, norm_log]
labels = ['WS-DETR', 'GroupNorm-WS-DETR']

HEAD_MAPPING = {
    1: 'Ti',
    2: 'S',
    3: 'B',
    4: 'L',
}

LAYER_MAPPING = {
    1: 'Ti',
    6: 'XL',
}

fig, ax = plt.subplots(figsize=(12, 5))
for entries, label, color in zip(logs, labels, colors):
    for h, ls in zip([1, 2, 3, 4], ['-', '--', ':', '-.']):
        epochs, ap = extract_test_ap(entries, h)
        ax.plot(epochs, ap, color=color, linestyle=ls, linewidth=1.5,
                label=f'{label} — {HEAD_MAPPING.get(h, h)}')

ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax.set_ylabel('AP@[.50:.95]', fontsize=12, fontweight='bold')
#ax.set_title('AP@[.50:.95] over Training', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
 77.43-75.59